In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

chinese_fonts = ['Microsoft YaHei', 'SimHei', 'STHeiti', 'SimSun', 'Arial Unicode MS']
sns.set_theme(style="white", rc={"font.sans-serif": chinese_fonts})
plt.rcParams['font.sans-serif'] = chinese_fonts
plt.rcParams['axes.unicode_minus'] = False

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 ADS 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_historical_hot_houses(city_name):
    """
    获取指定城市所有曾上榜的高关注房源（去重）。
    为了让 KDE 曲线平滑饱满，提取该城市一段时间内的所有热门房源。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        house_id,
        MAX(total_price) as total_price,
        MAX(unit_price) as unit_price
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily
    WHERE city = '{city_name}' 
      AND total_price > 0 
      AND unit_price > 0
    GROUP BY house_id
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_kde_distributions(city):
    """绘制高关注房源价格分布的核密度估计图"""
    df = fetch_historical_hot_houses(city)
    
    # KDE 需要一定的样本量才能算出平滑的曲线
    if df.empty or len(df) < 5:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, f"{city} 历史高关注房源样本过少 (当前: {len(df)}套)，无法拟合平滑密度曲线", 
                 ha='center', va='center', fontsize=14, color='#7F8C8D')
        plt.axis('off')
        plt.show()
        return

    # --- 1. 画布与子图布局 ---
    # 采用 1行2列的布局，左边总价，右边单价
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'[{city}] 市场高热度房源价格“引力中心”分布画像', fontsize=18, fontweight='bold', y=1.05)

    # --- 2. 绘制【总价】核密度图 (左侧) ---
    ax_total = axes[0]
    color_total = '#8E44AD' 
    p99_total = df['total_price'].quantile(0.99)
    df_total = df[df['total_price'] <= p99_total]
    
    sns.kdeplot(
        data=df_total, x='total_price', 
        ax=ax_total, fill=True, color=color_total, 
        alpha=0.5, linewidth=2, gridsize=200
    )
    
    # 添加中位数参考线
    median_total = df_total['total_price'].median()
    ax_total.axvline(median_total, color='#2C3E50', linestyle='--', linewidth=1.5, alpha=0.8)
    ax_total.text(median_total, ax_total.get_ylim()[1]*0.9, f' 中位数: {median_total:.1f}万', 
                  color='#2C3E50', fontweight='bold', va='top')

    ax_total.set_title('热门房源：总价分布密度', fontsize=14, pad=15)
    ax_total.set_xlabel('房源总价 (万元)', fontsize=12)

    # --- 3. 绘制【单价】核密度图 (右侧) ---
    ax_unit = axes[1]
    color_unit = '#16A085'  # 质感的深海绿
    
    p99_unit = df['unit_price'].quantile(0.99)
    df_unit = df[df['unit_price'] <= p99_unit]
    
    sns.kdeplot(
        data=df_unit, x='unit_price', 
        ax=ax_unit, fill=True, color=color_unit, 
        alpha=0.5, linewidth=2, gridsize=200
    )
    
    median_unit = df_unit['unit_price'].median()
    ax_unit.axvline(median_unit, color='#2C3E50', linestyle='--', linewidth=1.5, alpha=0.8)
    ax_unit.text(median_unit, ax_unit.get_ylim()[1]*0.9, f' 中位数: {median_unit:,.0f}元/㎡', 
                 color='#2C3E50', fontweight='bold', va='top')

    ax_unit.set_title('热门房源：单价分布密度', fontsize=14, pad=15)
    ax_unit.set_xlabel('房源单价 (元/平米)', fontsize=12)
    # 单价的 X 轴应用千分位格式化
    ax_unit.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

    for ax in axes:
        ax.set_ylabel('')
        ax.set_yticks([])
        sns.despine(ax=ax, left=True, top=True, right=True)
        ax.tick_params(axis='x', color='#BDC3C7')

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据。")
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_kde_distributions(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_kde_distributions(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=1, layout=Layout(width='250px'), options=('上海', '北京', '南京', '天津', '太原', …

Output()